# 2 · How a model learns

From a straight line to a neural network, all trained by one loop. Background and derivations are in the [notes](notes/02_how_a_model_learns.md).

Names as in notebook 1: input $x$ (the wind now), target $y$ (the wind $h$ hours later), forecast $\hat y$, the line $\hat y = w\,x + b$, and the **loss** $L$, the mean squared error on the training hours. Notebook 1 forecast the speed; from here on we forecast the components u and v, which the network of section 5 needs, starting with u alone (why: see the notes).

The tools, and the data: ERA5 wind shipped in `data/`, Bratislava 2020–2024 (up to 2023 for training, 2024 for testing) and Košice 2024 as a town the models never see.

In [ ]:
import math
import numpy as np
import torch
from windlab import fetch_wind, field_samples, fit, least_squares, mlp, patch, predict, rmse, warmup_cosine, year
from plots import (hardest_week, plot_curve_fit, plot_descent, plot_error_by_change, plot_forecasts, plot_one_step,
                   plot_optimizers, plot_training)

time, u, v = fetch_wind(*patch(48.15, 17.11), "2020-01-01", "2024-12-31", cache="data/bratislava_patch_2020-2024.npz")
time_ke, u_ke, v_ke = fetch_wind(*patch(48.72, 21.26), "2024-01-01", "2024-12-31", cache="data/kosice_patch_2024.npz")
centre = u.shape[1] // 2
u_ba = u[:, centre].astype(float)  # u at Bratislava (the centre grid point), in m/s

## 1 · Gradient descent

Most models have no formula for their best parameters. **Gradient descent** finds them by walking downhill on the loss: start anywhere, here at $w = b = 0$, and repeat a small step against the **gradient**, the slopes of the loss,

$$\frac{\partial L}{\partial w} = \frac2N\sum(\hat y - y)\,x, \qquad \frac{\partial L}{\partial b} = \frac2N\sum(\hat y - y),$$

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b},$$

where the **learning rate** $\eta$ sets the step size.

The code fits the line one hour ahead at Bratislava in 2023: first with notebook 1's formula, for reference, then with 200 steps of gradient descent. The plot: left, the line after 0, 1, 2, 5, 20 and 200 steps; right, the same steps on the loss landscape, where every grey contour joins lines of equal loss and the star is the formula's answer.

In [ ]:
def best_line(x, y):
    # Notebook 1's least-squares formula
    w = np.mean((x - x.mean()) * (y - y.mean())) / np.mean((x - x.mean()) ** 2)
    return w, y.mean() - w * x.mean()

def loss(w, b):
    return np.mean((w * x + b - y) ** 2)

def gradient(w, b, x, y):
    error = w * x + b - y  # forecast minus truth, for every hour
    return 2 * np.mean(error * x), 2 * np.mean(error)  # dL/dw, dL/db

hours = np.flatnonzero(year(time[:-1]) == 2023)
x, y = u_ba[hours], u_ba[hours + 1]  # x = u now, y = u one hour later
w_best, b_best = best_line(x, y)

lr = 0.05  # learning rate
w, b = 0.0, 0.0
path_gd = [(w, b)]
for step in range(200):
    dw, db = gradient(w, b, x, y)
    w, b = w - lr * dw, b - lr * db
    path_gd.append((w, b))

print(f"least squares:    w = {w_best:.4f}, b = {b_best:.4f}")
print(f"gradient descent: w = {w:.4f}, b = {b:.4f} after 200 steps")
plot_descent(x, y, path_gd, loss, best=(w_best, b_best))

**Look for:** big steps first, then a slow creep along the flat valley floor to the formula's answer.

**One step instead of 200.** Why so many steps? Take a bowl in one parameter, $L = \tfrac\kappa2\,(w - w^*)^2$. Its **curvature** $\kappa = d^2L/dw^2$, the second derivative (the slope of the slope), says how sharply it bends. The slope is $dL/dw = \kappa\,(w - w^*)$, so every step shrinks the distance to the bottom by the same factor,

$$w - w^* \;\leftarrow\; (1 - \eta\kappa)\,(w - w^*),$$

and the step size $\eta = 1/\kappa$ lands at the bottom in one step: $w \leftarrow w - \frac1\kappa\,\frac{dL}{dw}$.

With two parameters, $\theta = (w, b)$, the second derivatives form a 2 × 2 matrix, the **Hessian**

$$H = \begin{pmatrix} \partial^2 L/\partial w^2 & \partial^2 L/\partial w\,\partial b \cr \partial^2 L/\partial b\,\partial w & \partial^2 L/\partial b^2 \end{pmatrix} = 2\begin{pmatrix} \overline{x^2} & \bar x \cr \bar x & 1 \end{pmatrix}.$$

Along each of its two main axes the bowl is a one-parameter bowl as above, and the two curvatures $\kappa$ are the **eigenvalues** of $H$. For u, $\overline{x^2}$ is several times larger than 1, so the bowl bends sharply along $w$ and gently along $b$: it is long and thin, and no single $\eta$ suits both directions. Two ways out:

- **Newton's step** replaces the division by $\kappa$ with the inverse of $H$: $\;\theta \leftarrow \theta - H^{-1}\nabla L$, with the gradient $\nabla L = (\partial L/\partial w,\, \partial L/\partial b)$. Along each main axis this divides the slope by that axis's own curvature. Gradient descent uses only the first derivatives: it is a **first-order** method, and Newton's step, which needs $H$, is a **second-order** one.
- **Rescaling the input** to mean 0 and standard deviation 1 turns $H$ into $2\begin{pmatrix}1 & 0 \cr 0 & 1\end{pmatrix}$: both curvatures are 2, the bowl is round, and one plain step with $\eta = 1/2$ is enough. This works for a single input; with many correlated inputs it does not (see the notes).

The code computes $H$ and its curvatures, then tries both shortcuts from $w = b = 0$. The plot: left, the original bowl with the 200 steps above and one Newton step; right, the round bowl of the rescaled input and its single step.

In [ ]:
H = 2 * np.array([[np.mean(x * x), np.mean(x)], [np.mean(x), 1.0]])  # the Hessian: the second derivatives of L
print(f"H = {H.round(2).tolist()}; the curvatures, its eigenvalues: {np.linalg.eigvalsh(H).round(2).tolist()}")
w_newton, b_newton = -np.linalg.solve(H, gradient(0.0, 0.0, x, y))  # one Newton step from w = b = 0

x_scaled = (x - x.mean()) / x.std()  # the input rescaled to mean 0 and standard deviation 1
dw, db = gradient(0.0, 0.0, x_scaled, y)
w_scaled, b_scaled = -0.5 * dw, -0.5 * db  # one plain gradient step with lr = 1/2, from w = b = 0

print(f"least squares:                  w = {w_best:.4f}, b = {b_best:.4f}")
print(f"one Newton step:                w = {w_newton:.4f}, b = {b_newton:.4f}")
print(f"one step on the rescaled input: w = {w_scaled / x.std():.4f}, b = {b_scaled - w_scaled * x.mean() / x.std():.4f}"
      "  (in the original units)")
plot_one_step(path_gd, loss, (w_best, b_best), (w_newton, b_newton), x_scaled, y, (w_scaled, b_scaled))

**Look for:** both shortcuts land exactly on the least-squares line. But Newton's step is a **second-order** method, and for networks **rarely feasible**. With $n$ parameters, $H$ has $n \times n$ entries and takes about $n^3$ operations to solve: for the 5,122 parameters of the network in section 5, one Newton step would cost more than that network's whole training, and for a million parameters $H$ alone would fill 8 TB. And the loss of a network is no clean bowl, so the step can even point the wrong way. Newton's step is used where it is cheap, for linear models: `least_squares` in section 6 solves the same equations. Networks are trained by first-order methods such as AdamW (section 2).

## 2 · SGD and AdamW

A gradient step needs the gradient over all the training data. Real data sets often do not fit in memory and are read from disk in pieces, so gradient descent gets only one step per pass. **Stochastic gradient descent (SGD)** steps after every random **batch** instead, here 64 hours: one pass over the data gives 137 steps instead of one, at the price of noise. **AdamW**, the usual choice for neural networks, adds momentum $m$ (an average of recent gradients), a step size of its own for each parameter (the step is divided by $\sqrt{s}$, the typical size of its recent gradients), and a pull towards zero (weight decay $\lambda$), which keeps a network from leaning on large weights that fit noise in the training data. At step $k$:

$$m \leftarrow \beta_1 m + (1-\beta_1)\,g, \qquad s \leftarrow \beta_2 s + (1-\beta_2)\,g^2, \qquad \theta \leftarrow \theta - \eta\lambda\,\theta - \eta\,\frac{m/(1-\beta_1^k)}{\sqrt{s/(1-\beta_2^k)}+\epsilon},$$

with $\theta = (w, b)$, $g$ the gradient on the batch, and PyTorch's defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\lambda = 0.01$, $\epsilon = 10^{-8}$. Dividing by $1 - \beta^k$ corrects $m$ and $s$ for starting at zero.

The code draws 200 random batches of 64 hours and walks 200 steps with each method, written by hand: SGD with the gradient of section 1, AdamW with the three lines above.

In [ ]:
rng = np.random.default_rng(0)
batches = [rng.integers(0, len(x), size=64) for _ in range(200)]  # 64 random hours for each step

w, b = 0.0, 0.0
path_sgd = [(w, b)]
for i in batches:
    dw, db = gradient(w, b, x[i], y[i])
    w, b = w - lr * dw, b - lr * db
    path_sgd.append((w, b))

theta, m, s = np.zeros(2), np.zeros(2), np.zeros(2)  # theta = (w, b)
path_adamw = [tuple(theta)]
for k, i in enumerate(batches, start=1):
    g = np.array(gradient(*theta, x[i], y[i]))
    m = 0.9 * m + 0.1 * g
    s = 0.999 * s + 0.001 * g**2
    theta = theta - lr * 0.01 * theta - lr * (m / (1 - 0.9**k)) / (np.sqrt(s / (1 - 0.999**k)) + 1e-8)
    path_adamw.append(tuple(theta))

Then the three walks are compared, and their final lines are scored on the training year, on a new year and in a new town. The plot: left, the three paths on the loss landscape; middle, the error on 2024 after every step; right, the same error against the hours of data read so far. The table: the RMSE of every final line, one hour ahead.

In [ ]:
test_hours = np.flatnonzero(year(time[:-1]) == 2024)
data = {  # x = u now, y = u one hour later
    "2023 (training)": (x, y),
    "2024 (new year)": (u_ba[test_hours], u_ba[test_hours + 1]),
    "Kosice 2024 (new town)": (u_ke[:-1, centre], u_ke[1:, centre]),
}
paths = {"gradient descent": path_gd, "SGD": path_sgd, "AdamW": path_adamw}
plot_optimizers(paths, {"gradient descent": len(x), "SGD": 64, "AdamW": 64}, data["2024 (new year)"], loss, (w_best, b_best))

print(f"{'RMSE (m/s)':18s}" + "".join(f"{name:>24s}" for name in data))
for name, (w, b) in {"least squares": (w_best, b_best), **{n: p[-1] for n, p in paths.items()}}.items():
    print(f"{name:18s}" + "".join(f"{rmse(w * xs + b, ys):24.3f}" for xs, ys in data.values()))

**Look for:** per step, SGD follows gradient descent and then jitters; per hour of data read, it is near the bottom before gradient descent has finished one pass. AdamW is the slowest per step (per hour of data read it still beats gradient descent): it moves $w$ and $b$ by about $\eta$ each step, though $b$ needs to move only 0.016, so $b$ overshoots. Its strength is elsewhere: its step does not depend on the size of the gradient, so one $\eta$ works where SGD's would blow up (exercise 3). On new data all lines score within 0.015 m/s: for this line, the optimizer changes the cost, not what is learned.

## 3 · Autodiff

For thousands of parameters nobody derives gradients by hand: PyTorch records the computation and differentiates it automatically (**autodiff**). This loop is all the training there is: the networks below are trained by the same loop, in `windlab.fit`.

The code first asks PyTorch for the gradient at $w = b = 0$ and compares it with ours. Then `torch.nn.Linear(1, 1)`, our line, is trained by `torch.optim.AdamW` on the same batches, and its path is compared with the hand-written one.

In [ ]:
x_torch, y_torch = torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
w = torch.tensor(0.0, requires_grad=True)  # "track gradients for this number"
b = torch.tensor(0.0, requires_grad=True)
((w * x_torch + b - y_torch) ** 2).mean().backward()  # autodiff: fills w.grad and b.grad
dw, db = gradient(0.0, 0.0, x, y)
print(f"dL/dw, dL/db at w = b = 0: PyTorch {w.grad.item():.4f}, {b.grad.item():.4f}; by hand {dw:.4f}, {db:.4f}")

model = torch.nn.Linear(1, 1)  # one input, one output: w is model.weight, b is model.bias
torch.nn.init.zeros_(model.weight)
torch.nn.init.zeros_(model.bias)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
path_torch = [(0.0, 0.0)]
for i in batches:
    loss_torch = ((model(x_torch[i, None]) - y_torch[i, None]) ** 2).mean()
    optimizer.zero_grad()  # forget the previous gradient
    loss_torch.backward()  # autodiff
    optimizer.step()  # the AdamW update
    path_torch.append((model.weight.item(), model.bias.item()))
print(f"largest difference from the hand-written AdamW path: {np.abs(np.subtract(path_torch, path_adamw)).max():.1e}")

## 4 · A curve without a formula

The weight $w$ of the best line, the wind's memory (notebook 1, section 4), falls with the horizon $h$; here for u. Fit the curve $w(h) = e^{-h/\tau}$ to the 48 weights $w_h$: its one parameter, the **memory time** $\tau$, has no formula for this loss, so we walk downhill on

$$L(\tau) = \frac1{48}\sum_{h=1}^{48}\left(e^{-h/\tau} - w_h\right)^2, \qquad \frac{dL}{d\tau} = \frac2{48}\sum_{h=1}^{48}\left(e^{-h/\tau} - w_h\right)e^{-h/\tau}\,\frac{h}{\tau^2}.$$

The code measures the 48 weights with notebook 1's formula (horizons 1 to 48 hours, 2023), walks 100 steps downhill from $\tau = 3$ hours, and checks the result: at the end the slope is zero, and PyTorch computes the same slope as our formula. The plot: left, the weights (dots) and the curve after 0, 1, 2, 5, 10 and 100 steps; right, the loss against $\tau$ (grey) and the steps on it.

In [ ]:
horizons = np.arange(1, 49)
train_hours = np.flatnonzero(year(time[:-48]) == 2023)
w_h = np.array([best_line(u_ba[train_hours], u_ba[train_hours + h])[0] for h in horizons])  # notebook 1's formula, for u

def curve_loss(tau):
    return np.mean((np.exp(-horizons / tau) - w_h) ** 2)

def curve_gradient(tau):
    curve = np.exp(-horizons / tau)
    return np.mean(2 * (curve - w_h) * curve * horizons / tau**2)  # dL/dtau

lr_tau = 500  # the slope is tiny, so the step needs a large learning rate
tau = 3.0
path_tau = [tau]
for step in range(100):
    tau = tau - lr_tau * curve_gradient(tau)
    path_tau.append(tau)

tau_torch = torch.tensor(3.0, dtype=torch.float64, requires_grad=True)
((torch.exp(-torch.tensor(horizons) / tau_torch) - torch.tensor(w_h)) ** 2).mean().backward()

print(f"gradient descent: tau = {tau:.3f} h, where the slope is {curve_gradient(tau):.1e}")
print(f"slope at 3 h: by hand {curve_gradient(3.0):.6f}, PyTorch {tau_torch.grad.item():.6f}")
plot_curve_fit(horizons, w_h, path_tau, curve_loss)

**Look for:** a memory time of about 21 hours, where the slope of the loss is zero.

## 5 · A small neural network

The same loop trains a **multilayer perceptron (MLP)**: lines with a bend (GELU) after each, 5,122 parameters instead of 2.

$$z_1 = \mathrm{GELU}(W_1 \tilde x + b_1), \qquad z_2 = \mathrm{GELU}(W_2 z_1 + b_2), \qquad f(x;\theta) = W_3 z_2 + b_3,$$

with $\tilde x$ the input rescaled to mean 0 and standard deviation 1, as in section 1.

- **Input** $x$: the last 6 hours of u and v at Bratislava; **target** $y$: u and v 6 hours later.
- **Delta form** $\hat y = \text{wind now} + f(x;\theta)$: the network $f$ learns only the change from now, so it starts as persistence. We train it and the plain, **absolute** form $\hat y = f(x;\theta)$.
- **Train** on Bratislava 2020–2023; **test** on 2024 and in Košice, a town the model never saw (**zero-shot**).

The code builds the samples: for every hour $t$, the 12 inputs, the target 6 hours later, and the wind now; the same for Košice.

In [ ]:
LAGS, HORIZON = 6, 6
FIELD = False  # True: the whole 3 x 3 field around the town as input, not just its centre point

def samples(u, v):
    return field_samples(u, v, LAGS, HORIZON) if FIELD else field_samples(u[:, [centre]], v[:, [centre]], LAGS, HORIZON)

X_ba, Y_ba, now_ba, t_ba = samples(u, v)  # inputs x, targets y, the wind now (= persistence), the hour t
X_ke, Y_ke, now_ke, _ = samples(u_ke, v_ke)
train, test = year(time[t_ba]) <= 2023, year(time[t_ba]) == 2024

Then both networks are trained with the loop of section 3 (`windlab.fit`): AdamW on batches of 1,024 hours, for 10 **epochs**, that is 10 passes over all training hours, each in a new random order. The learning rate rises over the first 5 % of the steps and then falls smoothly to zero (**warm-up** and **cosine** schedule, see the notes). The plot shows their training loss after every step, with the loss of persistence as a dashed line.

In [ ]:
def train_mlp(target, epochs=10, batch_size=1024, seed=0):
    torch.manual_seed(seed)
    model = mlp(X_ba[train], n_out=2)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    steps = epochs * math.ceil(train.sum() / batch_size)
    losses = fit(model, X_ba[train], target, opt, epochs=epochs, batch_size=batch_size,
                 scheduler=warmup_cosine(opt, steps), seed=seed)
    return model, losses

absolute, losses_absolute = train_mlp(Y_ba[train])
delta, losses_delta = train_mlp(Y_ba[train] - now_ba[train])  # learns only the change from now
plot_training({"MLP absolute": losses_absolute, "MLP delta": losses_delta},
              mse_persistence=np.mean((now_ba[train] - Y_ba[train]) ** 2), steps_per_epoch=math.ceil(train.sum() / 1024))

**Look for:** the delta network starts at the persistence line; the absolute one needs about one and a half epochs just to get there.

## 6 · Forecasts against the truth, and a new town

The code adds **linear**, least squares on the same 12 inputs, and prints the RMSE and the skill of every forecast, in Bratislava 2024 and in Košice.

In [ ]:
W, b = least_squares(X_ba[train], Y_ba[train])
forecasts = {  # each takes the inputs X and the wind now, and returns the forecast
    "persistence": lambda X, now: now,
    "linear": lambda X, now: X @ W + b,
    "MLP absolute": lambda X, now: predict(absolute, X),
    "MLP delta": lambda X, now: now + predict(delta, X),
}
cases = {"Bratislava 2024": (X_ba[test], Y_ba[test], now_ba[test]), "Kosice 2024, zero-shot": (X_ke, Y_ke, now_ke)}
print(f"{'RMSE in m/s (skill)':20s}" + "".join(f"{case:>26s}" for case in cases))
for name, forecast in forecasts.items():
    scores = [(rmse(forecast(X, now), Y), rmse(now, Y)) for X, Y, now in cases.values()]  # model, persistence
    print(f"{name:20s}" + "".join(f"{r:17.3f} ({1 - (r / r0) ** 2:+.3f})" for r, r0 in scores))

The week of 2024 in which persistence misses most, chosen as in notebook 1: the truth (black) and three forecasts of u, 6 hours ahead.

In [ ]:
week = np.flatnonzero(test)[hardest_week(now_ba[test, 0] - Y_ba[test, 0])]
plot_forecasts(time[t_ba[week] + HORIZON], Y_ba[week, 0],
               {name: forecasts[name](X_ba[week], now_ba[week])[:, 0] for name in ("persistence", "linear", "MLP delta")},
               ylabel=f"y: u in {HORIZON} h (m/s)", title="Bratislava 2024: the week in which persistence does worst")

The same forecasts over all of 2024, grouped by how much the wind changed in those 6 hours; below each group, its share of the hours. That change is exactly the error of persistence, so its bars are set by the grouping itself: compare the line and the network with each other.

In [ ]:
plot_error_by_change({name: forecasts[name](X_ba[test], now_ba[test]) for name in ("persistence", "linear", "MLP delta")},
                     Y_ba[test], now_ba[test])

**Look for:** every forecast turns after the truth, and every error grows with the size of the change; the network beats the line only slightly. In Košice all lose skill: the delta network less than the absolute one, and the plain linear model least, so that it scores best there (with these inputs; exercise 2 and the notes show where that changes).

## Your turn (after the workshop)

1. In section 1, try `lr = 0.2` and `lr = 0.01`. What happens?
2. **Zero-shot:** keep `HORIZON = 6`, train only on Bratislava, and raise the skill in Košice: try `FIELD = True`, other `LAGS`, another network. Does the delta form keep its advantage?
3. In section 2, add `lr = 0.2` at the top of the first cell and run section 2 again. Which walk blows up, and why does AdamW not?

More exercises, and the reasons behind every step, are in the [notes](notes/02_how_a_model_learns.md).